# 02 National and regional analysis

Costruzione degli indicatori nazionali e regionali. Il notebook separa parametri, ricostruzione degli output, controlli e lettura dei risultati.


## Parametri modificabili

`REBUILD_INDICATORS` ricostruisce gli indicatori. `SNAPSHOT` può fissare uno snapshot. `TOP_N` regola le classifiche. `MIN_FUNDING` filtra le regioni con importi molto piccoli.


In [ ]:
REBUILD_INDICATORS = True
SNAPSHOT = None
TOP_N = 20
MIN_FUNDING = 0


In [ ]:
from pathlib import Path
import sys
import pandas as pd
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts.config import INDICATORS_DATA_DIR
from scripts.src.build_pnrr_indicators import build_pnrr_indicators
from scripts.utils import latest_snapshot, read_csv
pd.options.display.float_format = '{:,.2f}'.format


## Costruzione degli indicatori

Gli indicatori nazionali non duplicano i progetti territorializzati. Gli indicatori regionali usano gli importi allocati.


In [ ]:
indicator_dir = build_pnrr_indicators(snapshot=SNAPSHOT) if REBUILD_INDICATORS else INDICATORS_DATA_DIR / (SNAPSHOT or latest_snapshot(INDICATORS_DATA_DIR))
indicator_dir


## Sintesi nazionale


In [ ]:
national = read_csv(indicator_dir / 'national_summary.csv')
national


## Regioni

La classifica usa `finanziamento_pnrr_allocated`. Modificare `MIN_FUNDING` e `TOP_N` nella cella parametri.


In [ ]:
regional = read_csv(indicator_dir / 'regional_summary.csv')
regional['finanziamento_pnrr_allocated'] = pd.to_numeric(regional.get('finanziamento_pnrr_allocated'), errors='coerce')
regional.loc[regional['finanziamento_pnrr_allocated'].fillna(0) >= MIN_FUNDING].sort_values('finanziamento_pnrr_allocated', ascending=False).head(TOP_N)


## Controlli

Questi controlli aiutano a verificare rapidamente righe prodotte e grandezze aggregate.


In [ ]:
pd.DataFrame([{'national_rows': len(national), 'regional_rows': len(regional), 'regional_funding_sum': regional['finanziamento_pnrr_allocated'].sum()}]).T.rename(columns={0: 'value'})
